# Jour 3 · Isolation Forest et maintenance prédictive


## Objectifs

- détecter une combinaison multivariée inhabituelle
- transformer un score en alerte exploitable
- relier détection, diagnostic et décision de maintenance

Isolation Forest cherche des observations faciles à isoler dans l'espace des variables. Il ne connaît ni le mot « panne » ni la causalité : il produit un score d'étrangeté à replacer dans le contexte métier.

![Nuage de comportements fréquents et points inhabituels isolés par quelques coupures](../assets/jour_03/02_isolation_forest_intuition.png)

*Dans plusieurs dimensions, une combinaison rare peut être anormale même si chaque variable prise séparément semble plausible.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_labeled.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.set_index("timestamp").sort_index()
features = ["temperature_c", "humidity_pct", "power_kw", "pressure_bar", "vibration_mm_s"]

In [ ]:
train_end = df.index.min() + pd.Timedelta(days=28)
normal_train = df.loc[(df.index < train_end) & (df["is_anomaly"] == 0), features]

detector = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=1,
)
detector.fit(normal_train)

df["anomaly_score"] = -detector.decision_function(df[features])
df["iforest_flag"] = (detector.predict(df[features]) == -1).astype(int)

test = df.loc[df.index >= train_end]
print("Précision :", round(precision_score(test["is_anomaly"], test["iforest_flag"], zero_division=0), 3))
print("Rappel    :", round(recall_score(test["is_anomaly"], test["iforest_flag"], zero_division=0), 3))
print("F1        :", round(f1_score(test["is_anomaly"], test["iforest_flag"], zero_division=0), 3))

![Signal de vibration et score d'étrangeté produits par Isolation Forest](../assets/jour_03/02_score_isolation_forest.png)

*Le modèle produit d'abord un score continu ; la frontière de décision le transforme ensuite en alerte binaire.*

In [ ]:
view = test.last("14D")
flagged = view["iforest_flag"] == 1
ax = view["vibration_mm_s"].plot(figsize=(13, 4), label="vibration")
ax.scatter(view.index[flagged], view.loc[flagged, "vibration_mm_s"], c=view.loc[flagged, "anomaly_score"], cmap="Reds", label="alertes", zorder=3)
ax.set_ylabel("mm/s")
ax.set_title("Alertes multivariées d'Isolation Forest")
ax.legend()
plt.show()

## De l'anomalie à la maintenance

La détection n'est qu'une étape : il faut regrouper les alertes, ajouter leur contexte, éviter les doublons et définir qui agit. Une alerte isolée peut déclencher une inspection ; une dérive persistante peut justifier une intervention planifiée.

![Chaîne transformant un score d'anomalie en action de maintenance](../assets/jour_03/02_anomalie_vers_maintenance.png)

*Une alerte exploitable rassemble score, contexte, sévérité et action recommandée avant d'arriver au technicien.*

In [ ]:
alerts = test.loc[test["iforest_flag"] == 1, features + ["anomaly_score"]].copy()
alerts["severity"] = np.select(
    [
        alerts["vibration_mm_s"] > 2.5,
        alerts["anomaly_score"] > alerts["anomaly_score"].quantile(0.75),
    ],
    ["critique", "élevée"],
    default="à surveiller",
)
alerts["recommended_action"] = np.where(
    alerts["severity"] == "critique",
    "inspection sous 4 h",
    "contrôle lors de la prochaine ronde",
)
alerts.tail(10)

### À vous de jouer — résumer les alertes par jour

Calculez pour chaque jour le nombre d'alertes, le score maximum et la vibration maximale. Affichez les jours les plus critiques en premier.

In [ ]:
# Écrivez votre code ici.
pass

### À vous de jouer — inspecter les faux positifs

Affichez les cinq alertes au score le plus élevé qui ne correspondent pas au label synthétique. Pourquoi faut-il les examiner plutôt que les supprimer automatiquement ?

In [ ]:
# Écrivez votre code ici.
pass

## Architecture minimale

`capteur → passerelle/broker → stockage temporel → calcul des variables → détecteur → gestion des alertes → technicien`

En production, on surveille aussi les données manquantes, la dérive des scores et le retour des techniciens.